In [61]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from time import time

## Example

In [ ]:
load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    default_headers={
        "HTTP-Referer": "https://localhost:8888", 
        "X-Title": "E-Commerce Portfolio App",
    }    
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    messages=[{"role": "user", "content": "Hello OpenRouter!"}]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


## Create syllabus json

In [ ]:
markdown_base_path = "../../official_doc/exam_syllabus/machine-learning-engineer-associate-02-domain{domain_id}.md"

markdown_contents = []
for domain_id in range(1, 5):
    markdown_file_path = markdown_base_path.format(domain_id=domain_id)
    with open(markdown_file_path, "r") as md_file:
        md_content = md_file.read()
        markdown_contents.append(md_content)

json_format = {
  "domain_id": "string",
  "domain_name": "string",
  "task_id": "string",
  "task_name": "string",
  "skill_id": "string",
  "skill_name": "string"
}

json_example = {
  "domain_id": "1",
  "domain_name": "Data Engineering",
  "task_id": "1",
  "task_name": "Data Collection",
  "skill_id": "1",
  "skill_name": "Extract data from data sources."
}

output = []
for markdown_content in markdown_contents:
    json_prompt = f"""
    You are provided with a markdown file containing the a content domain of AWS MLA-C02 Exam.
    Based on a content, create a json file with the following structure:
    {json_format}

    For example a skill item: + Skill 1.1.1: Extract data from data sources.
    The corresponding json object would be:

    {json_example}

    Restriction: Do not include any additional information or explanations in the output. Only provide the JSON representation of the skills extracted from the markdown content.
    
    Here is the markdown file:
    {markdown_content}
    """
    
    response = client.chat.completions.create(
        model="z-ai/glm-5.3",
        messages=[{"role": "user", "content": json_prompt}],
        response_format={"type": "json_object"}
    )

    response_content = response.choices[0].message.content
    output.append(response_content)
    

In [ ]:
final_output = []
for item in output:
    items = json.loads(item).get("skills", [])
    final_output += items
with open("aws_mla_c02_skills.json", "w") as json_file:
    json.dump(final_output, json_file, indent=4)
    

## Notes creation

In [34]:
with open("aws_mla_c02_skills.json", "r") as json_file:
    skills_data = json.load(json_file)


In [ ]:
note_system_prompt = """
You are an AWS Examination expert and you are responsible in providing accurate and detailed answers to questions related to AWS services, architecture, and best practices. 
Your responses should be clear, concise, and informative, catering to both beginners and experienced professionals in the field of cloud computing.
"""

note_prompt_1 = """
You are now responsible in create study notes for the AWS Certified Machine Learning - Associate (MLA-C02) exam.
You are provided with a single skill item from the AWS MLA-C02 exam syllabus. 
You are also provided with a AWS domain review notes regarding the task item.

Your task is to create a comprehensive study note for that skill item, including explanations, examples, and any relevant information that would help a candidate understand the concept thoroughly.
Provide scenario-based examples similar to exam questions to illustrate the application of the skill item in real-world scenarios.
Provide exam tips and traps related to the skill item as well.
Utilize tables, mermaid diagrams, and other visual aids to enhance the clarity of your explanations.
Use the notes, your expertise and web search to create a detailed study note for the skill item. 

Output: Notes should be in markdown format. Do not include any additional information or explanations in the output. Only provide the markdown representation of the study notes for the skill item.
Example of the header structure for the study notes:
# Task 1.1: {domain_name} - {task_name}
## Skill 1.1.1: {skill_name}

"""

note_prompt_2 = """
Skill item for notes generation: {skill_item}
Domain review notes: {domain_review_notes}
"""

domain_review_notes_base_path = "../../official_doc/domain_review_notes"

In [ ]:
notes_output = []

for skill in skills_data:
    start_time = time()
    with open(f"{domain_review_notes_base_path}/{skill.get("domain_id", "")}.{skill.get("task_id", "")}.md", "r") as drn_file:
        domain_review_notes = drn_file.read()
    note_prompt = note_prompt_2.format(skill_item=skill, domain_review_notes=domain_review_notes)
    
    response = client.chat.completions.create(
        model="deepseek/deepseek-v4-pro-0813",
        messages=[
            {"role": "system", "content": note_system_prompt},
            {"role": "user", "content": note_prompt_1},
            {"role": "user", "content": note_prompt}
        ],
        max_completion_tokens=16384
    )
    study_note_content = response.choices[0].message.content
    notes_output.append(study_note_content)
    
    note_output_base_path = "../notes_by_skills"
    output_path = f"{note_output_base_path}/{skill.get("domain_id", "")}.{skill.get("task_id", "")}.{skill.get("skill_id", "")}.md"
    with open(output_path, "w", encoding="utf-8") as notes_file:
        notes_file.write(study_note_content)
    
    print(f"Study notes for skill {skill.get('skill_id', '')} saved to {output_path}. Time taken: {round(time() - start_time, 2)} seconds.")


KeyboardInterrupt: 